iter_fastq: FASTQ reads.
sliding_window_trimmer: trims by quality.
run_pipeline: discards short reads (MINLEN), calculates GC and Phred, and writes the result.
qualities_as_int / mean_phred_read: calculate Phred.
gc_bases_in_sequence: calculates G+C for GC.
write_fastq_record, plus_line_safe: write the read to a file.

In [ ]:
import warnings
PHRED_OFFSET = 33
def iter_fastq(path, *, skip_incomplete_last=True):
    with open(path, encoding="ascii", errors="replace") as f:
        while True:
            h = f.readline()
            if not h:
                return
            seq = f.readline()
            plus = f.readline()
            qual = f.readline()
            if not seq or not plus or not qual:
                head = h.rstrip("\n\r")
                msg = (
                    f"Заголовок: {head[:120]!r}"
                )
                if skip_incomplete_last:
                    warnings.warn(msg)
                    return
                raise ValueError(msg)
            yield (
                h.rstrip("\n\r"),
                seq.rstrip("\n\r"),
                plus.rstrip("\n\r"),
                qual.rstrip("\n\r"),
            )


def write_fastq_record(f, header, seq, plus, qual):
    f.write(header + "\n")
    f.write(seq + "\n")
    f.write(plus + "\n")
    f.write(qual + "\n")
def qualities_as_int(sequence, quality, phred_offset=PHRED_OFFSET, zero_ns=True):
    out = []
    for i, ch in enumerate(quality):
        q = ord(ch) - phred_offset
        if zero_ns and (sequence[i] in "Nn"):
            q = 0
        out.append(q)
    return out


def mean_phred_read(sequence, quality, phred_offset=PHRED_OFFSET):
    q = qualities_as_int(sequence, quality, phred_offset, zero_ns=True)
    return sum(q) / len(q) if q else 0.0


def gc_bases_in_sequence(sequence):
    s = sequence.upper()
    return s.count("G") + s.count("C")

def sliding_window_trimmer(
    sequence,
    quality,
    window_length,
    required_quality,
    phred_offset=PHRED_OFFSET,
):
    quals = qualities_as_int(sequence, quality, phred_offset, zero_ns=True)
    n = len(sequence)
    total_need = required_quality * window_length

    if n < window_length:
        return None
    total = sum(quals[:window_length])
    if total < total_need:
        return None

    keep = n
    for i in range(0, n - window_length):
        total = total - quals[i] + quals[i + window_length]
        if total < total_need:
            keep = i + window_length
            break

    i = keep
    last_q = quals[i - 1]
    while last_q < required_quality and i > 1:
        i -= 1
        last_q = quals[i - 1]

    if i < 1:
        return None
    if i < n:
        return sequence[:i], quality[:i]
    return sequence, quality


def plus_line_safe(plus_line):
    return plus_line if (plus_line and plus_line[0] == "+") else "+"

def run_pipeline(
    input_path,
    output_path,
    *,
    window_length=5,
    required_quality=30.0,
    min_length=60,
):
    reads_in = 0
    after_sw = 0
    written = 0
    bases_written = 0
    gc_bases = 0
    sum_mean_phred = 0.0
    min_len = None
    max_len = None

    with open(output_path, "w", encoding="ascii", newline="\n") as out_f:
        for header, seq, plus, qual in iter_fastq(input_path):
            reads_in += 1
            trimmed = sliding_window_trimmer(
                seq, qual, window_length, required_quality, PHRED_OFFSET
            )
            if trimmed is None:
                continue
            after_sw += 1
            new_seq, new_qual = trimmed
            if len(new_seq) < min_length:
                continue

            write_fastq_record(
                out_f, header, new_seq, plus_line_safe(plus), new_qual
            )
            written += 1
            L = len(new_seq)
            bases_written += L
            gc_bases += gc_bases_in_sequence(new_seq)
            sum_mean_phred += mean_phred_read(new_seq, new_qual, PHRED_OFFSET)
            min_len = L if min_len is None else min(min_len, L)
            max_len = L if max_len is None else max(max_len, L)

    gc_pct = (100.0 * gc_bases / bases_written) if bases_written else 0.0
    avg_len = round(bases_written / written) if written else 0
    avg_phred = (sum_mean_phred / written) if written else 0.0

    return {
        "reads_in": reads_in,
        "reads_after_sw": after_sw,
        "reads_written": written,
        "min_len": min_len,
        "avg_len": avg_len,
        "max_len": max_len,
        "gc_pct": gc_pct,
        "mean_phred_of_means": avg_phred,
        "output_path": output_path,
    }


In [6]:

import os

INPUT = "reads.fastq.txt"  
OUTPUT = "filtered.fastq"
WINDOW = 5
QUAL = 30.0
MINLEN = 60

print("Working folder:", os.getcwd())
print("Files (up to 30):", os.listdir(".")[:30])

if not os.path.isfile(INPUT):
    guess = [
        x
        for x in os.listdir(".")
        if "fastq" in x.lower() or x.lower().endswith((".fq", ".txt"))
    ]

rep = run_pipeline(
    INPUT,
    OUTPUT,
    window_length=WINDOW,
    required_quality=QUAL,
    min_length=MINLEN,
)

print("Result")
print("Reads at the entrance:", rep["reads_in"])
print("After SLIDINGWINDOW:", rep["reads_after_sw"])
print("Recorded after MINLEN:", rep["reads_written"])
print("Length min / avg / max:", rep["min_len"], rep["avg_len"], rep["max_len"])
print(f"GC-content: {rep['gc_pct']:.2f}%")
print(f"Average of average Phred reads: {rep['mean_phred_of_means']:.2f}")
print("Exit:", rep["output_path"])

Working folder: /home/dandreeva
Files (up to 30): ['Untitled3.ipynb', 'Untitled2.ipynb', 'HW4.ipynb', 'covid_tra_filtered', '.config', '.bashrc', '.Renviron', 'TRA_Shannon_pre_post.png', 'Untitled.ipynb', 'repertoires_storage', 'uniprot_download.fasta', '.local', 'TRA_Shannon_pre_post_by_vaccine.png', 'Untitled1.ipynb', 'mirpy', '.bash_history', 'archive.tar.gz', 'diversity_indices_covid_tra.csv', '.python_history', 'ensembl_download_2.fasta', 'кластеры.ipynb', 'clonotype.ipynb', 'TRA_Delta_Shannon_by_vaccine.png', 'TRA_Chao1_pre_post.png', '.cache', 'TRA_Delta_Chao1_by_vaccine.png', '.jupyter', '.juphub', 'diversity_plots', 'ensembl_download_1.fasta']
Result
Reads at the entrance: 239366
After SLIDINGWINDOW: 224338
Recorded after MINLEN: 165315
Length min / avg / max: 60 134 151
GC-content: 43.18%
Average of average Phred reads: 36.53
Exit: filtered.fastq
